# 00 — Colab T4 Memory Pilot

**Purpose (issue #5):** verify that Grounding DINO Swin-T fits on a Colab T4 (16 GB) with **top-k=100** proposal limiting and on-disk feature caching, and decide whether fallback backbones (OWL-ViT, YOLO-World-small, YOLO11-small) are needed.

**Pre-registered decisions to make here:**
1. VRAM usage per image (detector pass + optional frozen encoder pass).
2. Wall-clock time for detection + feature extraction on a few images.
3. top-k=100 vs top-k=300 runtime/memory deltas.
4. Whether the full pipeline (extract → cache → prototypes → Mode A) fits in T4 memory.

> Save outputs **outside** the repository. Clear cell outputs before committing this notebook.

In [ ]:
# %%capture
!pip install -q transformers torch torchvision pyyaml tqdm opencv-python-headless
!nvidia-smi

In [ ]:
import time
import numpy as np

from uadapt.models.backbone_loader import load_backbone, limit_top_k

# Run from the repo root so `src` is importable, or `pip install -e .`
MODEL_CFG = {
    "name": "grounding_dino_swinT",
    "framework": "transformers",
    "checkpoint": "IDEA-Research/grounding-dino-tiny",
    "inference": {"box_threshold": 0.25, "text_threshold": 0.25, "device": "cuda", "top_k": 100},
}
classes = ["person"]  # LADD pilot
backbone = load_backbone(MODEL_CFG, device="cuda")

In [ ]:
# TODO: point at a few pilot images (data/raw, outside git)
images = []  # load 3-5 images here

t0 = time.time()
proposals = [backbone.predict(img, classes, image_id=f"pilot_{i}") for i, img in enumerate(images)]
dt = time.time() - t0
print(f"{len(images)} images in {dt:.1f}s -> {dt / max(len(images), 1):.2f}s/img")
for p in proposals[:1]:
    print("top-k proposals:", len(limit_top_k(p, 100)))
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## Decision log (fill in after running)

| Check | Result | Decision |
|-------|--------|----------|
| VRAM peak (detector) | | |
| VRAM peak (extract+cache) | | |
| top-k=100 OK? | | |
| top-k=300 OK? (ablation only) | | |
| fallback backbone needed? | | |

Log the outcome in `docs/change_log.md`.